# Project 2: Market-basket analysis

**Course:** Algorithms for Massive Data — Master in Data Science for Economics (A.Y. 2025/26)

**Project Description.**
The task is to implement a system finding frequent itemsets (aka market-basket analysis), using the IMDB Movies Dataset, released under the Public Domain license. Specifically, each movie is to be considered a basket, and actors listed under the Star1, Star2, Star3 and Star4 fields as items. At least one of the techniques studied in the course should be applied.

**Student:** Matteo Hasa - **ID:** 56269A

## 1. Environment setup

In [1]:
# Install the official Kaggle API client used to download the dataset.
!pip install -q kaggle

In [2]:
# Install Java, download Spark and the findspark helper.
!apt-get update -qq
!apt-get install -y openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!pip install -q findspark

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [3]:
import os
os.environ["JAVA_HOME"]  = "/usr/lib/jvm/java-8-openjdk-amd64"   # JVM location
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"    # Spark location

In [4]:
# Initialise Spark and obtain a SparkContext (sc), the entry point for RDDs.
import findspark
findspark.init()                                                 # put pyspark on the path
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").getOrCreate()    # use all local cores
sc = spark.sparkContext                                          # RDD entry point for SON
print("Spark version:", spark.version)

Spark version: 3.5.0


In [5]:
# Standard library + pandas imports used throughout the notebook.
import time
import zipfile
from math import ceil
from itertools import combinations
from collections import defaultdict
import pandas as pd

## 2. Configuration


In [6]:
USE_FULL_DATASET = True     # True -> all baskets; False -> subsample
SUBSAMPLE_SIZE   = 300      # baskets kept when USE_FULL_DATASET is "False"
RANDOM_SEED      = 42       # fixes the subsample for reproducibility

MIN_SUPPORT      = 0.003    # fractional support (~3 movies out of 1000)
MIN_CONFIDENCE   = 0.50     # minimum confidence for a reported rule

SON_N_CHUNKS     = 4        # number of Spark partitions used by SON
PCY_NUM_BUCKETS  = 5000     # size of the PCY hash table

## 3. Download the dataset from Kaggle

In [7]:
os.environ['KAGGLE_USERNAME'] = "*****"
os.environ['KAGGLE_KEY']      = "*****"

DATASET_ID = "harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows"
!kaggle datasets download -d {DATASET_ID}

Dataset URL: https://www.kaggle.com/datasets/harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows
License(s): CC0-1.0
100% 175k/175k [00:00<00:00, 353kB/s]



In [8]:
# Unzip the archive next to the notebook.
zip_name = DATASET_ID.split('/')[-1] + ".zip"
with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall('imdb_data')
    csv_name = 'imdb_data/' + zf.namelist()[0]
print("Extracted file:", csv_name)

Extracted file: imdb_data/imdb_top_1000.csv


## 4. Load and inspect the data

In [9]:
df = pd.read_csv(csv_name)                                # load the CSV with pandas
print("Rows, columns:", df.shape)                         # expect (1000, 16)
df[['Series_Title', 'Star1', 'Star2', 'Star3', 'Star4']].head()

Rows, columns: (1000, 16)


,Series_Title,Star1,Star2,Star3,Star4
0,The Shawshank Redemption,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler
1,The Godfather,Marlon Brando,Al Pacino,James Caan,Diane Keaton
2,The Dark Knight,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine
3,The Godfather: Part II,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton
4,12 Angry Men,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler


## 5. From rows to baskets


In [12]:
def build_baskets(dataframe):
    '''Turn the movies dataframe into a list of baskets (each a set of actors).'''
    star_cols = ['Star1', 'Star2', 'Star3', 'Star4']     # columns holding the items
    baskets = []                                          # output list of baskets
    for _, row in dataframe[star_cols].iterrows():        # iterate over movies
        actors = set()                                    # one basket = a set of actors
        for col in star_cols:                            # look at each star column
            value = row[col]                             # raw cell value
            if pd.notna(value):                          # skip missing actors
                name = str(value).strip()               # normalise whitespace
                if name:                                 # skip empty strings
                    actors.add(name)                     # add actor to the basket
        if len(actors) >= 2:                             # keep only minable baskets
            baskets.append(actors)
    return baskets

all_baskets = build_baskets(df)
print("Total usable baskets:", len(all_baskets))
print("Example basket:", all_baskets[0])

Total usable baskets: 1000
Example basket: {'Tim Robbins', 'Bob Gunton', 'William Sadler', 'Morgan Freeman'}


In [14]:
def select_baskets(baskets):
    '''Apply the global subsampling switch and return the baskets to analyse.'''
    if USE_FULL_DATASET:                                  # global flag from section 2
        return baskets                                    # use every basket
    import random                                         # reproducible subsample
    rng = random.Random(RANDOM_SEED)
    size = min(SUBSAMPLE_SIZE, len(baskets))
    return rng.sample(baskets, size)

baskets = select_baskets(all_baskets)                    # working set of baskets
N_BASKETS = len(baskets)                                 # number of baskets analysed
print("Baskets used in this run:", N_BASKETS)

Baskets used in this run: 1000


## 6. Apriori (from scratch, all itemset sizes)


In [15]:
def count_singletons(baskets, support_count):
    '''First pass: frequent single items as {frozenset({item}): count}.'''
    counts = defaultdict(int)                             # item -> #baskets containing it
    for basket in baskets:                                # scan every basket once
        for item in basket:                              # and every item
            counts[item] += 1                            # increment its counter
    return {frozenset([item]): c                          # wrap each item in a 1-itemset
            for item, c in counts.items() if c >= support_count}

In [16]:
def generate_candidates(prev_frequent, k):
    '''Build (and prune) size-k candidates from frequent (k-1)-itemsets.'''
    prev_list = list(prev_frequent)                       # index the frequent (k-1)-itemsets
    candidates = set()                                    # collected size-k candidates
    for i in range(len(prev_list)):                       # every ordered pair...
        for j in range(i + 1, len(prev_list)):            # ...of (k-1)-itemsets
            union = prev_list[i] | prev_list[j]           # their union is a candidate seed
            if len(union) == k:                           # keep only if it has size k
                # Pruning: every (k-1)-subset of 'union' must be frequent.
                if all(frozenset(sub) in prev_frequent
                       for sub in combinations(union, k - 1)):
                    candidates.add(union)
    return candidates

In [17]:
def count_itemsets(baskets, candidates):
    '''Count, for each candidate itemset, how many baskets contain it.'''
    counts = defaultdict(int)                             # candidate -> support count
    for basket in baskets:                                # one pass over the data
        for cand in candidates:                          # test each candidate
            if cand <= basket:                           # subset test (cand ⊆ basket)
                counts[cand] += 1
    return counts

In [18]:
def apriori(baskets, support_count, max_k=None):
    '''Full Apriori: {frozenset: count} for every frequent itemset.'''
    frequent = {}                                         # collects ALL frequent itemsets
    current = count_singletons(baskets, support_count)    # level 1: frequent items
    frequent.update(current)
    current_sets = set(current.keys())                    # frequent (k-1)-itemsets so far
    k = 2
    while current_sets and (max_k is None or k <= max_k): # levels k = 2, 3, ...
        candidates = generate_candidates(current_sets, k) # build + prune candidates
        if not candidates:
            break
        counts = count_itemsets(baskets, candidates)      # count over the data
        current = {iset: c for iset, c in counts.items() if c >= support_count}
        if not current:                                   # no frequent k-itemset
            break
        frequent.update(current)
        current_sets = set(current.keys())                # become the new "previous"
        k += 1
    return frequent

## 7. PCY (Park–Chen–Yu) for frequent pairs

In [19]:
def pcy(baskets, support_count, num_buckets=5000):
    '''PCY algorithm returning frequent pairs as {frozenset(pair): count}.'''
    # First pass: count items and hash pairs into buckets
    item_counts = defaultdict(int)                        # item -> count
    buckets = [0] * num_buckets                           # bucket -> #pairs hashed to it
    for basket in baskets:
        items = list(basket)                              # materialise the basket
        for it in items:                                  # count single items
            item_counts[it] += 1
        for i in range(len(items)):                       # hash every pair in the basket
            for j in range(i + 1, len(items)):
                pair = tuple(sorted((items[i], items[j]))) # canonical (a,b) with a<b
                buckets[hash(pair) % num_buckets] += 1     # increment its bucket

    # Between passes: frequent items + frequent-bucket bitmap
    freq_items = {it for it, c in item_counts.items() if c >= support_count}
    bitmap = [1 if buckets[b] >= support_count else 0      # 1 if bucket is frequent
              for b in range(num_buckets)]
    del buckets                                            # bitmap replaces the counters

    # Second pass: count only surviving candidate pairs
    pair_counts = defaultdict(int)
    for basket in baskets:
        items = [it for it in basket if it in freq_items]  # keep frequent items only
        for i in range(len(items)):
            for j in range(i + 1, len(items)):
                pair = tuple(sorted((items[i], items[j])))
                if bitmap[hash(pair) % num_buckets] == 1:  # PCY extra condition
                    pair_counts[pair] += 1
    return {frozenset(p): c for p, c in pair_counts.items() if c >= support_count}

## 8. SON on Apache Spark

In [20]:
def son_spark(sc, baskets, min_support_fraction, n_chunks):
    '''SON on Spark: returns {frozenset: global_count} of frequent itemsets.'''
    n = len(baskets)                                      # total number of baskets
    global_support = min_support_fraction * n             # absolute global threshold
    chunk_size = ceil(n / n_chunks)                       # baskets per partition
    chunks = [baskets[i:i + chunk_size]                   # split data into chunks
              for i in range(0, n, chunk_size)]

    # Pass 1 (map): local Apriori on each partition -> candidate itemsets
    def local_apriori(chunk):
        local_support = min_support_fraction * len(chunk) # threshold scaled to the chunk
        return list(apriori(chunk, local_support).keys()) # local frequent itemsets

    candidates = (sc.parallelize(chunks, n_chunks)        # one task per chunk
                    .flatMap(local_apriori)               # emit local frequent itemsets
                    .distinct()                           # union (drop duplicates)
                    .collect())                           # bring candidates to the driver
    candidates = set(candidates)
    bc = sc.broadcast(candidates)                         # broadcast to every worker

    # PASS 2 (reduce): global count of candidates over all baskets
    def emit_contained(basket):                           # map: (candidate, 1) if present
        return [(c, 1) for c in bc.value if c <= basket]

    result = (sc.parallelize(baskets)                     # distribute the baskets
                .flatMap(emit_contained)                  # emit candidate occurrences
                .reduceByKey(lambda a, b: a + b)          # sum counts per candidate
                .filter(lambda kv: kv[1] >= global_support) # keep globally frequent
                .collect())
    return dict(result)

## 9. Run the analysis and inspect the frequent itemsets

In [21]:
support_count = MIN_SUPPORT * N_BASKETS                  # absolute support threshold
print(f"Min support {MIN_SUPPORT} -> >= {support_count:.1f} baskets out of {N_BASKETS}")

# SON on Spark is a scalable solution; it returns frequent itemsets of all sizes.
t0 = time.time()
frequent_itemsets = son_spark(sc, baskets, MIN_SUPPORT, SON_N_CHUNKS)
print(f"SON (Spark) found {len(frequent_itemsets)} frequent itemsets in {time.time()-t0:.2f}s")

Min support 0.003 -> >= 3.0 baskets out of 1000
SON (Spark) found 299 frequent itemsets in 108.53s


In [22]:
# Tidy, sorted table of the frequent itemsets (multi-actor groups first).
rows = []
for itemset, count in frequent_itemsets.items():
    rows.append({'size': len(itemset),
                 'actors': ', '.join(sorted(itemset)),
                 'support_count': count,
                 'support': count / N_BASKETS})
freq_df = (pd.DataFrame(rows)
           .sort_values(['size', 'support_count'], ascending=[False, False])
           .reset_index(drop=True))
freq_df.head(20)

,size,actors,support_count,support
0,3,"Daniel Radcliffe, Emma Watson, Rupert Grint",5,0.005
1,3,"Carrie Fisher, Harrison Ford, Mark Hamill",3,0.003
2,3,"Elijah Wood, Ian McKellen, Orlando Bloom",3,0.003
3,2,"Daniel Radcliffe, Rupert Grint",6,0.006
4,2,"Daniel Radcliffe, Emma Watson",5,0.005
5,2,"Emma Watson, Rupert Grint",5,0.005
6,2,"Tim Allen, Tom Hanks",4,0.004
7,2,"Joe Pesci, Robert De Niro",4,0.004
8,2,"Harrison Ford, Mark Hamill",3,0.003
9,2,"Carrie Fisher, Harrison Ford",3,0.003


## 10. Correctness: the three algorithms agree


In [23]:
# Ground-truth Apriori over the whole dataset (no partitioning).
direct = apriori(baskets, support_count)
direct_pairs = {iset for iset in direct if len(iset) == 2}   # its frequent pairs

# PCY frequent pairs and SON itemsets.
pcy_pairs_set = set(pcy(baskets, support_count, PCY_NUM_BUCKETS).keys())
son_sets      = set(frequent_itemsets.keys())

assert son_sets == set(direct.keys()), "SON vs Apriori mismatch!"
assert pcy_pairs_set == direct_pairs,  "PCY vs Apriori (pairs) mismatch!"
print("OK - Apriori == SON (all sizes), and PCY == Apriori (pairs).")

OK - Apriori == SON (all sizes), and PCY == Apriori (pairs).


## 11. From frequent itemsets to association rules


In [24]:
def association_rules(frequent_itemsets, n_baskets, min_confidence):
    '''Generate association rules from the mined frequent itemsets.'''
    rules = []
    for itemset, iset_count in frequent_itemsets.items():
        if len(itemset) < 2:                              # rules need >= 2 items
            continue
        items = list(itemset)
        for r in range(1, len(items)):                    # antecedent size 1..|itemset|-1
            for antecedent in combinations(items, r):
                A = frozenset(antecedent)                 # antecedent
                C = itemset - A                           # consequent
                A_count = frequent_itemsets.get(A)        # supp(A)
                C_count = frequent_itemsets.get(C)        # supp(C)
                if not A_count:
                    continue
                confidence = iset_count / A_count         # P(C | A)
                if confidence < min_confidence:
                    continue
                lift = confidence / (C_count / n_baskets) if C_count else float('nan')
                rules.append({'antecedent': ', '.join(sorted(A)),
                              'consequent': ', '.join(sorted(C)),
                              'support': iset_count / n_baskets,
                              'confidence': confidence,
                              'lift': lift})
    return rules

rules = association_rules(frequent_itemsets, N_BASKETS, MIN_CONFIDENCE)
rules_df = (pd.DataFrame(rules)
            .sort_values(['lift', 'confidence'], ascending=False)
            .reset_index(drop=True))
print(f"Generated {len(rules_df)} rules (min_confidence={MIN_CONFIDENCE}).")
rules_df.head(20)

Generated 46 rules (min_confidence=0.5).


,antecedent,consequent,support,confidence,lift
0,Mark Hamill,"Carrie Fisher, Harrison Ford",0.003,1.000000,333.333333
1,"Carrie Fisher, Harrison Ford",Mark Hamill,0.003,1.000000,333.333333
2,Elijah Wood,"Ian McKellen, Orlando Bloom",0.003,1.000000,333.333333
3,"Ian McKellen, Orlando Bloom",Elijah Wood,0.003,1.000000,333.333333
4,"Harrison Ford, Mark Hamill",Carrie Fisher,0.003,1.000000,250.000000
5,Elijah Wood,Orlando Bloom,0.003,1.000000,250.000000
6,"Elijah Wood, Ian McKellen",Orlando Bloom,0.003,1.000000,250.000000
7,Mark Hamill,Carrie Fisher,0.003,1.000000,250.000000
8,Carrie Fisher,"Harrison Ford, Mark Hamill",0.003,0.750000,250.000000
9,Orlando Bloom,Elijah Wood,0.003,0.750000,250.000000


## 12. Scalability experiment



In [25]:
import random
fractions = [0.3, 0.6, 1.0]                               # growing data fractions
results = []
for frac in fractions:
    random.seed(RANDOM_SEED)                              # reproducible subset
    sub = random.sample(baskets, int(N_BASKETS * frac))   # take a fraction of baskets
    sc_thr = MIN_SUPPORT * len(sub)                       # absolute threshold for this size

    t = time.perf_counter(); apriori(sub, sc_thr);                       t_ap = time.perf_counter()-t
    t = time.perf_counter(); pcy(sub, sc_thr, PCY_NUM_BUCKETS);          t_pcy = time.perf_counter()-t
    t = time.perf_counter(); son_spark(sc, sub, MIN_SUPPORT, SON_N_CHUNKS); t_son = time.perf_counter()-t

    results.append({'fraction': frac, 'n_baskets': len(sub),
                    'apriori_s': round(t_ap, 4),
                    'pcy_s': round(t_pcy, 4),
                    'son_spark_s': round(t_son, 4)})
pd.DataFrame(results)

,fraction,n_baskets,apriori_s,pcy_s,son_spark_s
0,0.3,300,52.1541,0.0054,6.6284
1,0.6,600,7.9485,0.0047,25.8712
2,1.0,1000,9.4826,0.0073,100.2357


## 13. Comments and discussion

**Data & encoding.** Only the `Star1`–`Star4` columns are used; each movie is a
basket of actor names. Set membership is the natural representation for market-basket
mining, so no numeric encoding is required.

**Three algorithms, one result.** Apriori gives the methodological backbone
(downward-closure pruning) and mines itemsets of any size. PCY targets Apriori's real
bottleneck — counting candidate pairs — by hashing pairs into buckets and pruning via
a bucket bitmap. SON turns the in-memory algorithm into a distributed, two-pass Spark
job with no false negatives. Section 10 verifies that all three agree on the same
inputs, which is our correctness guarantee.

**Scalability.** Baskets contain at most 4 actors, so itemsets never exceed size 4 and
candidate generation is cheap; the cost is dominated by the linear counting passes. On
1000 movies Apriori/PCY are fastest because they avoid Spark's scheduling overhead, but
they are bounded by single-machine memory. SON is the solution that scales: its Pass-1
work is local to each partition and Pass-2 is a single distributed aggregation, so it
runs unchanged on clusters and on data that exceeds RAM.

**Results.** With a low support threshold the frequent multi-actor itemsets are
dominated by franchise casts and recurring on-screen partnerships (e.g. trilogies that
reuse the same leads); the high-lift rules pinpoint actor pairs that co-star far more
often than chance — the actionable output of the analysis.

**Relation to existing work.** The implementation is original. Unlike notebooks that
call `mlxtend.apriori`, Apriori, PCY and SON are all written from scratch, SON runs on
real Spark, and the three are cross-validated against each other.
